In [3]:
# Banking Marketing Campaign Analytics Dashboard
# Initial Dataset Inspection
# Banking Dataset - Marketing Targets
# Source: https://www.kaggle.com/datasets/prakharrathi25/banking-dataset-marketing-targets
# Purpose:
# Load and inspect the banking marketing dataset to understand:
#   - Dataset structure
#   - Column names
#   - Data types
#   - Number of rows and columns
#   - Sample records
#   - Missing values
#   - Duplicate records
#   - Potential data quality issues

# 1. Import required libraries

import pandas as pd
import numpy as np
from google.colab import files

# 2. Upload the dataset files

print("Upload train.csv and test.csv")
uploaded = files.upload()

print("\nFiles uploaded successfully:")
for filename in uploaded.keys():
    print("-", filename)

# 3. Identify train and test files

train_file = None
test_file = None

for filename in uploaded.keys():
    name = filename.lower()

    if "train" in name:
        train_file = filename
    elif "test" in name:
        test_file = filename


if train_file is None:
    raise FileNotFoundError("Training CSV file was not found.")

if test_file is None:
    raise FileNotFoundError("Test CSV file was not found.")

# 4. Load the datasets
# The Kaggle files use semicolons (;) as separators.

train_df = pd.read_csv(train_file, sep=";")
test_df = pd.read_csv(test_file, sep=";")

print("\nDatasets loaded successfully.")


# 5. Dataset dimensions

print("\n" + "=" * 70)
print("DATASET DIMENSIONS")
print("=" * 70)

print(f"Training dataset rows    : {train_df.shape[0]:,}")
print(f"Training dataset columns : {train_df.shape[1]}")

print(f"\nTest dataset rows        : {test_df.shape[0]:,}")
print(f"Test dataset columns     : {test_df.shape[1]}")


# 6. Display column names

print("\n" + "=" * 70)
print("COLUMN NAMES")
print("=" * 70)

for number, column in enumerate(train_df.columns, start=1):
    print(f"{number:2}. {column}")

# 7. Display sample rows

print("\n" + "=" * 70)
print("FIRST 5 ROWS OF TRAINING DATA")
print("=" * 70)

display(train_df.head())


print("\n" + "=" * 70)
print("FIRST 5 ROWS OF TEST DATA")
print("=" * 70)

display(test_df.head())


# 8. Inspect data types

print("\n" + "=" * 70)
print("COLUMN DATA TYPES")
print("=" * 70)

data_type_summary = pd.DataFrame({
    "Column": train_df.columns,
    "Data Type": train_df.dtypes.astype(str).values
})

display(data_type_summary)

# 9. General dataset information

print("\n" + "=" * 70)
print("TRAINING DATASET INFORMATION")
print("=" * 70)

train_df.info()

# 10. Check standard missing values

print("\n" + "=" * 70)
print("STANDARD MISSING VALUES (NaN)")
print("=" * 70)

missing_summary = pd.DataFrame({
    "Column": train_df.columns,
    "Missing Values": train_df.isnull().sum().values,
    "Missing Percentage": (
        train_df.isnull().sum().values / len(train_df) * 100
    ).round(2)
})

display(missing_summary)

total_missing = train_df.isnull().sum().sum()

print(f"\nTotal standard missing values: {total_missing:,}")

# 11. Check duplicate rows

print("\n" + "=" * 70)
print("DUPLICATE ROW CHECK")
print("=" * 70)

train_duplicates = train_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()

print(f"Duplicate rows in training dataset: {train_duplicates:,}")
print(f"Duplicate rows in test dataset    : {test_duplicates:,}")


# 12. Inspect categorical values

print("\n" + "=" * 70)
print("CATEGORICAL COLUMN VALUES")
print("=" * 70)

categorical_columns = train_df.select_dtypes(include="object").columns

for column in categorical_columns:

    print(f"\nColumn: {column}")
    print("-" * 50)

    value_counts = train_df[column].value_counts(dropna=False)

    print(value_counts)


# 13. Check for 'unknown' values

# In this dataset, some missing or unavailable information is
# represented by the text "unknown" instead of NaN.

print("\n" + "=" * 70)
print("'UNKNOWN' VALUE CHECK")
print("=" * 70)

unknown_results = []

for column in categorical_columns:

    unknown_count = (
        train_df[column]
        .astype(str)
        .str.lower()
        .eq("unknown")
        .sum()
    )

    unknown_percentage = (unknown_count / len(train_df)) * 100

    unknown_results.append({
        "Column": column,
        "Unknown Count": unknown_count,
        "Unknown Percentage": round(unknown_percentage, 2)
    })


unknown_summary = pd.DataFrame(unknown_results)

# Only display columns that actually contain "unknown"
unknown_summary = unknown_summary[
    unknown_summary["Unknown Count"] > 0
].reset_index(drop=True)

display(unknown_summary)

# 14. Inspect numeric columns

print("\n" + "=" * 70)
print("NUMERIC COLUMN SUMMARY")
print("=" * 70)

display(train_df.describe().T)

# 15. Check numeric ranges

print("\n" + "=" * 70)
print("NUMERIC VALUE RANGES")
print("=" * 70)

numeric_columns = train_df.select_dtypes(include=np.number).columns

range_summary = pd.DataFrame({
    "Column": numeric_columns,
    "Minimum": [train_df[col].min() for col in numeric_columns],
    "Maximum": [train_df[col].max() for col in numeric_columns],
    "Mean": [
        round(train_df[col].mean(), 2)
        for col in numeric_columns
    ],
    "Median": [
        round(train_df[col].median(), 2)
        for col in numeric_columns
    ]
})

display(range_summary)

# 16. Inspect special pdays value

print("\n" + "=" * 70)
print("PDAYS SPECIAL VALUE CHECK")
print("=" * 70)

pdays_minus_one = (train_df["pdays"] == -1).sum()
pdays_minus_one_pct = pdays_minus_one / len(train_df) * 100

print(
    f"Rows where pdays = -1: "
    f"{pdays_minus_one:,} "
    f"({pdays_minus_one_pct:.2f}%)"
)

print(
    "\nNote: pdays = -1 is a special value indicating that "
    "the customer was not previously contacted."
)


# 17. Check negative balances

print("\n" + "=" * 70)
print("NEGATIVE BALANCE CHECK")
print("=" * 70)

negative_balances = (train_df["balance"] < 0).sum()

print(f"Customers with negative account balance: {negative_balances:,}")

print(
    "\nNegative balances are not automatically errors because "
    "bank accounts may have overdrafts or negative balances."
)


# 18. Inspect target variable

print("\n" + "=" * 70)
print("TARGET VARIABLE: y")
print("=" * 70)

target_counts = train_df["y"].value_counts()

target_percentages = (
    train_df["y"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percentages
})

display(target_summary)

print(
    "\n'y' represents whether the customer subscribed "
    "to the bank's term deposit."
)


# 19. Compare train and test structure

print("\n" + "=" * 70)
print("TRAIN / TEST STRUCTURE CHECK")
print("=" * 70)

same_columns = list(train_df.columns) == list(test_df.columns)

print("Train and test have identical columns:", same_columns)

if same_columns:
    print("Both datasets contain the same 17 variables.")
else:
    print("Warning: Train and test dataset columns differ.")

# 20. Create final inspection summary

print("\n" + "=" * 70)
print("INITIAL DATA INSPECTION SUMMARY")
print("=" * 70)

print(f"""
Dataset successfully loaded into pandas.

TRAINING DATASET
----------------
Rows    : {train_df.shape[0]:,}
Columns : {train_df.shape[1]}

TEST DATASET
------------
Rows    : {test_df.shape[0]:,}
Columns : {test_df.shape[1]}

DATA TYPES
----------
Integer columns : {len(train_df.select_dtypes(include='integer').columns)}
Object/Text columns : {len(train_df.select_dtypes(include='object').columns)}

DATA QUALITY
------------
Standard NaN missing values : {train_df.isnull().sum().sum():,}
Duplicate training rows     : {train_duplicates:,}
Duplicate test rows         : {test_duplicates:,}

Potential quality issues identified:

1. Several categorical columns contain the value 'unknown'.
   These are not stored as NaN but may represent unavailable
   or missing information.

2. The 'pdays' variable contains -1 as a special value.
   This represents customers who were not previously contacted
   and should not automatically be treated as an error.

3. Some customers have negative account balances.
   These values should be reviewed but are plausible banking
   values and should not automatically be removed.

4. The target column 'y' contains 'yes' and 'no', indicating
   whether the customer subscribed to a term deposit.

5. No exact duplicate records were identified in either dataset.

The dataset structure is suitable for further cleaning,
exploratory data analysis, and development of the banking
marketing campaign analytics dashboard.
""")

# 21. Final column summary table

column_summary = pd.DataFrame({
    "Column": train_df.columns,
    "Data Type": train_df.dtypes.astype(str).values,
    "Missing Values": train_df.isnull().sum().values,
    "Unique Values": train_df.nunique().values
})

print("\n" + "=" * 70)
print("FINAL COLUMN SUMMARY")
print("=" * 70)

display(column_summary)

Upload train.csv and test.csv


Saving test.csv to test.csv
Saving train.csv to train.csv

Files uploaded successfully:
- test.csv
- train.csv

Datasets loaded successfully.

DATASET DIMENSIONS
Training dataset rows    : 45,211
Training dataset columns : 17

Test dataset rows        : 4,521
Test dataset columns     : 17

COLUMN NAMES
 1. age
 2. job
 3. marital
 4. education
 5. default
 6. balance
 7. housing
 8. loan
 9. contact
10. day
11. month
12. duration
13. campaign
14. pdays
15. previous
16. poutcome
17. y

FIRST 5 ROWS OF TRAINING DATA


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no



FIRST 5 ROWS OF TEST DATA


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no



COLUMN DATA TYPES


... (9 outputs left out)


PDAYS SPECIAL VALUE CHECK
Rows where pdays = -1: 36,954 (81.74%)

Note: pdays = -1 is a special value indicating that the customer was not previously contacted.

NEGATIVE BALANCE CHECK
Customers with negative account balance: 3,766

Negative balances are not automatically errors because bank accounts may have overdrafts or negative balances.

TARGET VARIABLE: y


,Count,Percentage
y,,
no,39922,88.3
yes,5289,11.7



'y' represents whether the customer subscribed to the bank's term deposit.

TRAIN / TEST STRUCTURE CHECK
Train and test have identical columns: True
Both datasets contain the same 17 variables.

INITIAL DATA INSPECTION SUMMARY

Dataset successfully loaded into pandas.

TRAINING DATASET
----------------
Rows    : 45,211
Columns : 17

TEST DATASET
------------
Rows    : 4,521
Columns : 17

DATA TYPES
----------
Integer columns : 7
Object/Text columns : 10

DATA QUALITY
------------
Standard NaN missing values : 0
Duplicate training rows     : 0
Duplicate test rows         : 0

Potential quality issues identified:

1. Several categorical columns contain the value 'unknown'.
   These are not stored as NaN but may represent unavailable
   or missing information.

2. The 'pdays' variable contains -1 as a special value.
   This represents customers who were not previously contacted
   and should not automatically be treated as an error.

3. Some customers have negative account balances.
   T

,Column,Data Type,Missing Values,Unique Values
0,age,int64,0,77
1,job,object,0,12
2,marital,object,0,3
3,education,object,0,4
4,default,object,0,2
5,balance,int64,0,7168
6,housing,object,0,2
7,loan,object,0,2
8,contact,object,0,3
9,day,int64,0,31


The Banking Marketing dataset was successfully loaded into pandas. The training dataset contains 45,211 records and 17 variables, while the test dataset contains 4,521 records and the same 17 variables. The dataset includes 7 numerical and 10 categorical columns. No standard null values or duplicate records were identified. However, several categorical attributes contain the value “unknown,” particularly contact and poutcome, which may represent unavailable information. The pdays field also uses -1 to indicate customers who were not previously contacted. These values should therefore be considered during later data-cleaning and analysis stages.

In [4]:
# Banking Marketing Campaign Analytics Dashboard
# Data Cleaning and Preprocessing
# Purpose:
# Prepare the banking marketing dataset for analysis and
# visualization by:
#   - Handling missing / unknown values
#   - Correcting data types
#   - Standardizing categorical values
#   - Handling special pdays values
#   - Creating useful analysis-ready fields


import pandas as pd
import numpy as np


# 1. Create copies of the original datasets

# Keeping the original datasets unchanged is useful in case
# we need to compare the cleaned data with the raw data later.

train_clean = train_df.copy()
test_clean = test_df.copy()

print("Copies of the original datasets created successfully.")


# 2. Standardize column names

# Convert column names to lowercase and remove unnecessary spaces.

train_clean.columns = (
    train_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

test_clean.columns = (
    test_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nColumn names standardized.")


# 3. Standardize categorical text

# Remove leading/trailing spaces and convert categorical
# values to lowercase for consistent formatting.

categorical_columns = train_clean.select_dtypes(include="object").columns

for column in categorical_columns:

    train_clean[column] = (
        train_clean[column]
        .str.strip()
        .str.lower()
    )

    test_clean[column] = (
        test_clean[column]
        .str.strip()
        .str.lower()
    )

print("Categorical text values standardized.")


# 4. Handle 'unknown' values

# Initial inspection showed that several columns contain
# "unknown" instead of standard NaN missing values.
#
# We will NOT guess or statistically impute these categories.
# Instead, they are renamed to "not_provided".
#
# This preserves the information that the original value
# was unavailable without introducing potentially incorrect data.

unknown_columns = [
    "job",
    "education",
    "contact",
    "poutcome"
]

for column in unknown_columns:

    train_clean[column] = train_clean[column].replace(
        "unknown",
        "not_provided"
    )

    test_clean[column] = test_clean[column].replace(
        "unknown",
        "not_provided"
    )


print("\n'unknown' values replaced with 'not_provided'.")


# 5. Handle the special pdays value

# In the original dataset:
# pdays = -1 means that the customer was NOT previously contacted.
#
# Instead of treating -1 as a normal number of days, we create a
# separate indicator showing whether previous contact occurred.
#
# Then pdays = -1 is changed to 0.
#
# Interpretation:
# previously_contacted = "no" and pdays = 0
# means there was no previous campaign contact.


train_clean["previously_contacted"] = np.where(
    train_clean["pdays"] == -1,
    "no",
    "yes"
)

test_clean["previously_contacted"] = np.where(
    test_clean["pdays"] == -1,
    "no",
    "yes"
)


train_clean["pdays"] = train_clean["pdays"].replace(-1, 0)
test_clean["pdays"] = test_clean["pdays"].replace(-1, 0)


print("Special pdays = -1 values handled.")


# 6. Improve poutcome interpretation

# If there was no previous contact, previous campaign outcome
# is more accurately represented as "not_previously_contacted"
# rather than simply "not_provided".

train_clean.loc[
    train_clean["previously_contacted"] == "no",
    "poutcome"
] = "not_previously_contacted"


test_clean.loc[
    test_clean["previously_contacted"] == "no",
    "poutcome"
] = "not_previously_contacted"


print("Previous campaign outcome standardized.")


# 7. Create numeric target variable

# Original target:
# y = yes / no
#
# Numeric version:
# yes = 1
# no  = 0
#
# The original y column is preserved because readable labels
# are useful for dashboard visualizations.

train_clean["target"] = train_clean["y"].map({
    "yes": 1,
    "no": 0
})

test_clean["target"] = test_clean["y"].map({
    "yes": 1,
    "no": 0
})


print("Numeric target variable created.")


# 8. Correct numeric data types

numeric_columns = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "target"
]


for column in numeric_columns:

    train_clean[column] = pd.to_numeric(
        train_clean[column],
        errors="coerce"
    )

    test_clean[column] = pd.to_numeric(
        test_clean[column],
        errors="coerce"
    )


print("Numeric data types verified.")


# 9. Assign categorical data types

category_columns = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "poutcome",
    "y",
    "previously_contacted"
]


for column in category_columns:

    train_clean[column] = train_clean[column].astype("category")

    test_clean[column] = test_clean[column].astype("category")


print("Categorical data types assigned.")


# 10. Set month as an ordered categorical variable

month_order = [
    "jan",
    "feb",
    "mar",
    "apr",
    "may",
    "jun",
    "jul",
    "aug",
    "sep",
    "oct",
    "nov",
    "dec"
]


train_clean["month"] = pd.Categorical(
    train_clean["month"],
    categories=month_order,
    ordered=True
)

test_clean["month"] = pd.Categorical(
    test_clean["month"],
    categories=month_order,
    ordered=True
)


print("Month converted to an ordered categorical variable.")


# 11. Check for missing values after cleaning

print("\n" + "=" * 70)
print("MISSING VALUES AFTER CLEANING")
print("=" * 70)


train_missing = train_clean.isnull().sum()

missing_table = pd.DataFrame({
    "Column": train_clean.columns,
    "Missing Values": train_missing.values
})

display(missing_table)


print(
    "\nTotal missing values after cleaning:",
    train_clean.isnull().sum().sum()
)



# 12. Check remaining 'unknown' values

print("\n" + "=" * 70)
print("UNKNOWN VALUE CHECK AFTER CLEANING")
print("=" * 70)


remaining_unknown = {}

for column in train_clean.select_dtypes(
    include=["object", "category"]
).columns:

    count = (
        train_clean[column]
        .astype(str)
        .str.lower()
        .eq("unknown")
        .sum()
    )

    remaining_unknown[column] = count


unknown_check = pd.DataFrame(
    list(remaining_unknown.items()),
    columns=["Column", "Remaining Unknown Values"]
)

display(unknown_check)



# 13. Check duplicates again


print("\n" + "=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

print(
    "Duplicate training records:",
    train_clean.duplicated().sum()
)

print(
    "Duplicate test records:",
    test_clean.duplicated().sum()
)



# 14. Display cleaned data types


print("\n" + "=" * 70)
print("FINAL DATA TYPES")
print("=" * 70)


data_type_summary = pd.DataFrame({
    "Column": train_clean.columns,
    "Data Type": train_clean.dtypes.astype(str).values
})

display(data_type_summary)



# 15. Display sample cleaned records


print("\n" + "=" * 70)
print("SAMPLE CLEANED DATA")
print("=" * 70)

display(train_clean.head(10))



# 16. Compare before and after

print("\n" + "=" * 70)
print("BEFORE VS AFTER CLEANING")
print("=" * 70)

print(
    f"Original training shape : {train_df.shape}"
)

print(
    f"Cleaned training shape  : {train_clean.shape}"
)

print(
    f"\nOriginal test shape     : {test_df.shape}"
)

print(
    f"Cleaned test shape      : {test_clean.shape}"
)



# 17. Final preprocessing summary

print("\n" + "=" * 70)
print("DATA CLEANING AND PREPROCESSING SUMMARY")
print("=" * 70)


print(f"""
DATA CLEANING COMPLETED SUCCESSFULLY

1. MISSING VALUES
-----------------
The initial dataset contained no standard NaN missing values.

However, several categorical variables used the value
'unknown' to represent unavailable information.

These values were standardized to 'not_provided' instead of
being statistically imputed, preventing unsupported assumptions
about customer characteristics.


2. PDAYS SPECIAL VALUE
----------------------
The original pdays variable used -1 to indicate that a customer
had not been contacted in a previous campaign.

A new variable named 'previously_contacted' was created.

    no  = customer was not previously contacted
    yes = customer was previously contacted

The -1 pdays values were then converted to 0.


3. DATA TYPES
-------------
Numeric variables were verified and converted to numeric types.

Categorical variables were converted to pandas category types.

The month variable was converted to an ordered categorical
variable from January through December.


4. CATEGORICAL STANDARDIZATION
------------------------------
Categorical values were converted to lowercase and unnecessary
spaces were removed.

The label 'unknown' was standardized to 'not_provided'.

For customers who had never been contacted previously,
poutcome was labeled 'not_previously_contacted'.


5. TARGET VARIABLE
------------------
The original target variable 'y' was preserved.

A numeric target variable was also created:

    yes = 1
    no  = 0


6. FINAL DATASET
----------------
Training rows    : {train_clean.shape[0]:,}
Training columns : {train_clean.shape[1]}

Test rows        : {test_clean.shape[0]:,}
Test columns     : {test_clean.shape[1]}

The cleaned datasets are now ready for exploratory data
analysis and dashboard visualization.
""")



# 18. Save cleaned datasets

train_clean.to_csv(
    "bank_marketing_train_clean.csv",
    index=False
)

test_clean.to_csv(
    "bank_marketing_test_clean.csv",
    index=False
)


print("Cleaned datasets saved successfully:")
print("- bank_marketing_train_clean.csv")
print("- bank_marketing_test_clean.csv")

Copies of the original datasets created successfully.

Column names standardized.
Categorical text values standardized.

'unknown' values replaced with 'not_provided'.
Special pdays = -1 values handled.
Previous campaign outcome standardized.
Numeric target variable created.
Numeric data types verified.
Categorical data types assigned.
Month converted to an ordered categorical variable.

MISSING VALUES AFTER CLEANING


,Column,Missing Values
0,age,0
1,job,0
2,marital,0
3,education,0
4,default,0
5,balance,0
6,housing,0
7,loan,0
8,contact,0
9,day,0



Total missing values after cleaning: 0

UNKNOWN VALUE CHECK AFTER CLEANING


,Column,Remaining Unknown Values
0,job,0
1,marital,0
2,education,0
3,default,0
4,housing,0
5,loan,0
6,contact,0
7,month,0
8,poutcome,0
9,y,0



DUPLICATE CHECK
Duplicate training records: 0
Duplicate test records: 0

FINAL DATA TYPES


,Column,Data Type
0,age,int64
1,job,category
2,marital,category
3,education,category
4,default,category
5,balance,int64
6,housing,category
7,loan,category
8,contact,category
9,day,int64



SAMPLE CLEANED DATA


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,previously_contacted,target
0,58,management,married,tertiary,no,2143,yes,no,not_provided,5,may,261,1,0,0,not_previously_contacted,no,no,0
1,44,technician,single,secondary,no,29,yes,no,not_provided,5,may,151,1,0,0,not_previously_contacted,no,no,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,not_provided,5,may,76,1,0,0,not_previously_contacted,no,no,0
3,47,blue-collar,married,not_provided,no,1506,yes,no,not_provided,5,may,92,1,0,0,not_previously_contacted,no,no,0
4,33,not_provided,single,not_provided,no,1,no,no,not_provided,5,may,198,1,0,0,not_previously_contacted,no,no,0
5,35,management,married,tertiary,no,231,yes,no,not_provided,5,may,139,1,0,0,not_previously_contacted,no,no,0
6,28,management,single,tertiary,no,447,yes,yes,not_provided,5,may,217,1,0,0,not_previously_contacted,no,no,0
7,42,entrepreneur,divorced,tertiary,yes,2,yes,no,not_provided,5,may,380,1,0,0,not_previously_contacted,no,no,0
8,58,retired,married,primary,no,121,yes,no,not_provided,5,may,50,1,0,0,not_previously_contacted,no,no,0
9,43,technician,single,secondary,no,593,yes,no,not_provided,5,may,55,1,0,0,not_previously_contacted,no,no,0



BEFORE VS AFTER CLEANING
Original training shape : (45211, 17)
Cleaned training shape  : (45211, 19)

Original test shape     : (4521, 17)
Cleaned test shape      : (4521, 19)

DATA CLEANING AND PREPROCESSING SUMMARY

DATA CLEANING COMPLETED SUCCESSFULLY

1. MISSING VALUES
-----------------
The initial dataset contained no standard NaN missing values.

However, several categorical variables used the value
'unknown' to represent unavailable information.

These values were standardized to 'not_provided' instead of
being statistically imputed, preventing unsupported assumptions
about customer characteristics.


2. PDAYS SPECIAL VALUE
----------------------
The original pdays variable used -1 to indicate that a customer
had not been contacted in a previous campaign.

A new variable named 'previously_contacted' was created.

    no  = customer was not previously contacted
    yes = customer was previously contacted

The -1 pdays values were then converted to 0.


3. DATA TYPES
------------